Public Score - 0.53314	
Private Score - 0.56278

In [1]:
# ==========================================
# CELL 1: SETUP & WEIGHT ATTACHMENT
# ==========================================
import os
import pandas as pd
from pathlib import Path
import torch

try:
    from ultralytics import YOLO
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
    from ultralytics import YOLO

# 1. Dataset Paths
INPUT_ROOT = Path("/kaggle/input/competitions/road-vision/RoadVision_DUET")
SAMPLE_SUB = INPUT_ROOT / "test" / "sample_submission.csv"
TEST_IMAGES  = INPUT_ROOT / "test" / "images"

# 2. THE BRAIN: Point this to the best.pt file from your 0.55 mAP run!
# Example: "/kaggle/input/roadvision-training-v1/best.pt"
# 2. THE BRAIN: Point this to the best.pt file from your 0.55 mAP run!
WEIGHTS_PATH = "/kaggle/input/notebooks/tahjibhossainkhan/duet-version-1/runs/exp/weights/best.pt"

print(f"Test Images mapped: {TEST_IMAGES}")
assert TEST_IMAGES.exists(), "Test images not found!"
assert os.path.exists(WEIGHTS_PATH), f"Weights not found at {WEIGHTS_PATH}. Please attach your previous notebook's output!"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Test Images mapped: /kaggle/input/competitions/road-vision/RoadVision_DUET/test/images


In [2]:
# ==========================================
# CELL 2: TTA INFERENCE LOOP
# ==========================================
print("Loading the optimized network matrices...")
model = YOLO(WEIGHTS_PATH)

# Index the unannotated test set
test_idx = {p.stem: p for p in TEST_IMAGES.rglob("*.*") if p.suffix.lower() in ['.jpg', '.png', '.jpeg']}
test_paths = sorted(list(test_idx.values()))

# We scale the image size up from 960 to 1280.
# This quadratically increases the spatial resolution, allowing the convolutional 
# kernels to map the geometries of highly distant vehicles.
IMGSZ = 1280

# We keep our AUC-maximizing ultra-low threshold
PRED_CONF = 0.001 
PRED_IOU = 0.60

predictions_dict = {}

print(f"Executing Test-Time Augmentation on {len(test_paths)} images...")
print("This applies horizontal flips and multiscale tensor variations on the fly.")

# Run the native Ultralytics prediction engine with augment=True
# This automatically handles the geometric inversions and WBF merging under the hood.
results = model.predict(
    source=test_paths, 
    imgsz=IMGSZ, 
    conf=PRED_CONF, 
    iou=PRED_IOU, 
    augment=True,       # <--- THIS IS THE TTA TRIGGER
    stream=True,        # Processes images one-by-one to prevent VRAM overflow
    verbose=False
)

for r in results:
    stem = Path(r.path).stem
    b = r.boxes
    pred_string_parts = []
    
    if b is not None and len(b) > 0:
        cls = b.cls.cpu().numpy().astype(int)
        conf = b.conf.cpu().numpy()
        xywhn = b.xywhn.cpu().numpy() # Extract normalized coordinate tensors
        
        # Sort predictions by descending confidence to ensure strict NMS hierarchy
        for i in conf.argsort()[::-1]:
            xc, yc, w, h = xywhn[i]
            
            # Map probabilities to the [0, 100] schema
            conf_percent = float(conf[i]) * 100.0
            
            pred_string_parts.extend([
                str(int(cls[i])), 
                f"{conf_percent:.2f}", 
                f"{xc:.6f}", f"{yc:.6f}", f"{w:.6f}", f"{h:.6f}"
            ])
            
    predictions_dict[stem] = " ".join(pred_string_parts)

print("TTA Forward Pass Complete!")

Loading the optimized network matrices...
Executing Test-Time Augmentation on 327 images...
This applies horizontal flips and multiscale tensor variations on the fly.
TTA Forward Pass Complete!


In [3]:
# ==========================================
# CELL 3: DIRECT MATRIX COMPILATION (FIXED)
# ==========================================
import pandas as pd

# 1. Bypass the truncated sample_submission entirely.
# Convert our fully populated predictions dictionary directly into a list of rows.
submission_rows = [{'image_id': stem, 'PredictionString': pred} for stem, pred in predictions_dict.items()]
final_df = pd.DataFrame(submission_rows)

# 2. Sort mathematically by image_id to guarantee alignment with Kaggle's backend evaluator
final_df = final_df.sort_values('image_id').reset_index(drop=True)

# 3. Save the full un-truncated matrix
output_csv = "/kaggle/working/submission_tta.csv"
final_df.to_csv(output_csv, index=False)

print(f"\nFinal TTA submission generated at {output_csv}")
print(f"Total Rows Compiled: {len(final_df)} (Should match your total test images!)")


Final TTA submission generated at /kaggle/working/submission_tta.csv
Total Rows Compiled: 327 (Should match your total test images!)
